In [ ]:
#!pip install dspy pydantic

In [ ]:
import dspy
from pydantic import BaseModel
from ftplib import FTP

In [ ]:
test_accessions = [
    "GSE174188",
    "GSE209912",
    "GSE188367",
    "GSE136103"
]

# Tools

In [ ]:
#!pip install scanpy

In [ ]:
import re
import warnings
import tempfile
import os
import tarfile
import scanpy as sc
import gzip
import shutil
import pandas as pd

def get_geo_ftp_path(accession: str) -> str:
    """
    Return the FTP directory for a GEO accession (GSE or GSM).
    """
    prefix = accession[:3]     # GSE or GSM
    number = accession[3:]
    chunk = prefix + number[:-3] + "nnn"

    # the ftp site stores series and samples in directories named by the accession number with the last three digits replaced by 'nnn'
    if prefix == "GSE":
        return f"/geo/series/{chunk}/{accession}/suppl/"
    elif prefix == "GSM":
        return f"/geo/samples/{chunk}/{accession}/suppl/"
    else:
        raise ValueError("Only GSE or GSM supported")

def list_geo_files(accession: str):

    ftp = FTP("ftp.ncbi.nlm.nih.gov")
    ftp.login()

    path = get_geo_ftp_path(accession)
    try:
        ftp.cwd(path)
    except:
        try:
            path = re.sub(r"suppl/$", "", path)
            ftp.cwd(path)
            warnings.warn(f"No supplementary files for: {accession}")
        except:
            raise FileNotFoundError(f"Could not find FTP path: {path}")

    files = ftp.nlst()
    ftp.quit()
    return files

def download_geo_supp_file(accession: str, file_name:str, output_dir: str):
    ftp = FTP("ftp.ncbi.nlm.nih.gov")
    ftp.login()
    
    path = get_geo_ftp_path(accession)
    try:
        ftp.cwd(path)
    except:
        try:
            path = re.sub(r"suppl/$", "", path)
            ftp.cwd(path)
            warnings.warn(f"No supplementary files for: {accession}")
        except:
            raise FileNotFoundError(f"Could not find FTP path: {path}")

    local_file_path = os.path.join(output_dir, file_name)
    with open(local_file_path, "wb") as f:
        try:
            ftp.retrbinary(f"RETR {file_name}", f.write)
        except Exception as e:
            ftp.quit()
            raise e
    ftp.quit()
    return local_file_path

def list_tar_contents(file_name: str):
     with tarfile.open(file_name, "r:*") as tar:
        for member in tar.getmembers():
            print(member.name)

def unpack_tar_file(tar_file_path: str, output_dir: str):
    with tarfile.open(tar_file_path, "r") as tar:
        tar.extractall(path=output_dir)

def build_anndata(counts_directory: str, sample_name: str):
    adata = sc.read_10x_mtx(counts_directory)
    
    # add sample name to obs and store anndata in dictionary
    adata.obs["sample_name"] = sample_name
    # adatas[sample_name] = adata
    
    # make a subdirectory to store anndata files
    adata_dir = os.path.join(counts_directory, "adatas")
    os.makedirs(adata_dir, exist_ok=True)
    sc.write_h5ad(adata, os.path.join(adata_dir, f"{sample_name}.h5ad"))

    # check that the file was actually saved
    saved_file_path = os.path.join(adata_dir, f"{sample_name}.h5ad")
    if os.path.exists(saved_file_path):
        print(f"Anndata object successfully saved at: {saved_file_path}")
    else:
        raise FileNotFoundError(f"Failed to save anndata object at: {saved_file_path}")
    
    return saved_file_path

# Rename files according to 10x Genomics conventions
def rename_geo_files(directory: str):
    files = os.listdir(directory)
    
    matrix = None
    features = None
    barcodes = None

    for f in files:
        n = f.lower()

        # matrix
        if "mtx" in n:
            matrix = f
            continue

        # features (genes)
        if any(x in n for x in ["gene", "feature", "symbol"]):
            features = f
            continue

        # barcodes (cells)
        if any(x in n for x in ["barcode", "cell"]):
            barcodes = f
            continue

    # Safety check
    if not (matrix and features and barcodes):
        raise ValueError(
            f"Could not find all required files in {directory}. "
            f"Found matrix={matrix}, features={features}, barcodes={barcodes}"
        )

    rename_map = {
        matrix: "matrix.mtx",
        features: "features.tsv",
        barcodes: "barcodes.tsv"
    }

    # check if files are gzipped and add appropriate extension to the new name
    for key, value in list(rename_map.items()):
        if key.endswith(".gz"):
            rename_map[key] = value + ".gz"

    # apply the new names by using a bash mv command
    for old, new in rename_map.items():
        src = os.path.join(directory, old)
        dst = os.path.join(directory, new)
        shutil.move(src, dst)
        print(f"Renamed {src} → {dst}")

    return directory

# structure 10x directory
def structure_10x_directory(directory):
    # create a new directory for the 10x files
    counts_directory = os.path.join(directory, "10x_counts")
    os.makedirs(counts_directory, exist_ok=True)

    # move the relevant files to the new directory
    for file in os.listdir(directory):
        if file in ["matrix.mtx", "matrix.mtx.gz", "features.tsv", "features.tsv.gz", "barcodes.tsv", "barcodes.tsv.gz"]:
            src = os.path.join(directory, file)
            dst = os.path.join(counts_directory, file)
            shutil.move(src, dst)
    
    return counts_directory

# convert csv files to tsv files
def convert_csv_to_tsv(file_path):
        
    # handle uncompressed csvs
    if file_path.endswith(".csv"):
        tsv_file_path = file_path[:-4] + ".tsv" # change suffix
        with open(file_path, "r") as csv_file, open(tsv_file_path, "w") as tsv_file:
            for line in csv_file:
                tsv_file.write(line.replace(",", "\t"))

    # handle gzipped csvs
    if file_path.endswith(".csv.gz"):
        tsv_file_path = file_path[:-7] + ".tsv.gz" # change suffix
        with gzip.open(file_path, "rt") as csv_file, gzip.open(tsv_file_path, "wt") as tsv_file:
            for line in csv_file:
                tsv_file.write(line.replace(",", "\t"))

    if not file_path.endswith(".csv.gz") and not file_path.endswith(".csv"):
        tsv_file_path = file_path  # no conversion needed
    return tsv_file_path

def format_features_file(features_file_path):
    
    # if file contains only one column, add a second column identical to the first with name gene_id
    if features_file_path.endswith(".gz"):
        with gzip.open(features_file_path, "rt") as f:
            lines = f.readlines()
    else:
        with open(features_file_path, "r") as f:
            lines = f.readlines()
    header = lines[0].strip().split("\t")
    rows = [line.strip().split("\t") for line in lines[1:]]

    features_df = pd.DataFrame(rows, columns=header)
    if features_df.shape[1] == 1:
        features_df["gene_id"] = features_df.iloc[:, 0]
        if features_file_path.endswith(".gz"):
            with gzip.open(features_file_path, "wt") as f:
                features_df.to_csv(f, sep="\t", index=False)
        else:
            with open(features_file_path, "w") as f:
                features_df.to_csv(f, sep="\t", index=False)

    return features_file_path

In [ ]:
tmpdir = tempfile.mkdtemp()
file_lists = {acc: list_geo_files(acc) for acc in test_accessions}
file_lists

In [ ]:
tar_file = "GSE188367_RAW.tar"

accession = "GSE188367"

download_geo_supp_file(accession, tar_file, tmpdir)
list_tar_contents(tmpdir + "/" + tar_file)

In [ ]:
unpack_tar_file(tmpdir + "/" + tar_file, tmpdir)

list_tar_contents(tmpdir + "/" + "GSM5678317_BM-Old4_counts.tar.gz")

# Pydantic data classes

In [ ]:
class GEO_entry(BaseModel):
    accession: str
    title: str
    summary: str
    overall_design: str
    contributor: str
    pubmed_ids: list
    supplementary_files: list

class expression_data(BaseModel):
    sample_id: str
    gene_ids: list
    counts: list
    accession: str

class anndata_object(BaseModel):
    adata: object
    accession: str

# DSPy Agents

In [ ]:
class DSPyGEOFetcher(dspy.Signature):
    """You are a computational biologist that goes through NCBI GEO uploads and fetches expression data and turns it into an anndata object.
    
    You are given an accession number and a set of tools. 
    You will go and retrieve data. 
    Sometimes it will be in tar files. 

    Then you will process the data files, converting and renaming as necessary. Then process into an anndata object, save it, and report the location.
    
    You must decide which tools are the best to handle the user's request."""

    user_request: str = dspy.InputField()
    directory: str = dspy.InputField(
        desc = (
                    "The temporary directory where all downloaded and processed files should be stored."
                )
    )
    accession: str = dspy.InputField(
        desc = (
                    "The GEO accession number (GSE or GSM) to fetch data for."
                )
    )
    process_result: str = dspy.OutputField(
        desc = (
                    "Message that summarizes the process result and the information retrieved."
                )
    )
        
    

In [ ]:
agent = dspy.ReAct(
    DSPyGEOFetcher,
    tools = [
        get_geo_ftp_path,
        list_geo_files,
        list_tar_contents,
        unpack_tar_file,
        download_geo_supp_file,
        convert_csv_to_tsv,
        rename_geo_files,
        format_features_file,
        build_anndata

    ]
)

In [ ]:
import sys
import os

In [ ]:
# get the Claude API key from local text file
# check if we're on MacOS or Windows and read appropriate file
if sys.platform.startswith("win"):
    with open("C:/Users/David/.claude_api.txt") as f:
        claude_key = f.read().strip()
else:
    with open("/Users/tatarakis/.api-keys/tatarakis-test-key.txt") as f:
        claude_key = f.read().strip()


In [ ]:
# define the language model to be used by all dspy agents in this notebook
lm = dspy.LM('anthropic/claude-sonnet-4-5-20250929', api_key=claude_key)
dspy.configure(lm=lm)

In [ ]:
test_message = lm(messages=[{"role": "user", "content": "Confirm that this test worked and we're ready to use Claude for agents. But do it as if you are Columbo when he suspects the user is the murderer but is still trying to be coy and disarming."}])  # => ['This is a test!']

print(test_message[0])

In [ ]:
test_accession = test_accessions[1]

In [ ]:
prediction = agent(user_request = "oh mighty agent, what is your purpose? Answer me like Columbo.")

In [ ]:
print(list(prediction.trajectory.values())[0])

In [ ]:
tmpdir = tempfile.mkdtemp()

In [ ]:
print(tmpdir)

In [ ]:
os.listdir(tmpdir)

In [ ]:
adatas = {}

In [ ]:
agent(user_request = (
    "I need you to download and process the data for accession " + 
    test_accession + 
    ". Take the following steps: " +
    "1) list and downlaod the supplemental files for that accession" +
    "2) if there's a tar file, unpack it. If there are more tar files, do this for them as well. " +
    "3) look for 10X files matrix, features, and barcodes. If there are any .csv or .csv.gz files, turn them into .tsv/.tsv.gz files."
    "4) rename the files to match the 10X specifications for scanpy's read_10x function. Save the prefix stripped off as sample_name if there is one" + 
    "5) call structure_10x_directory(directory)" +
    "6) take the returned counts_directory" +
    "7) call build_anndata(counts_directory, sample_name)" +
    "8) if that fails, try to format the features file by adding a second column identical to the first with name gene_id, then " +
    "9) call build_anndata(counts_directory, sample_name)" +
    "After organizing the files, you MUST call structure_10x_directory(directory). Then you MUST call build_anndata(counts_directory, sample_name) using the returned counts_directory. Do not stop until these tools have been executed. Return the output path of build_anndata."
    ), 
    directory = tmpdir,
    accession = test_accession
)

In [ ]:
os.listdir(tmpdir)

In [ ]:
os.listdir(tmpdir + "/10x_counts")

In [ ]:
tmpdir

In [ ]:
adatas

In [ ]:
rename_geo_files(directory=tmpdir, accession="GSE209912")

In [ ]:
os.listdir(tmpdir)